<!-- ============================================================ -->
<!-- NOTEBOOK HEADER — MLOps Introductory Course on GCP           -->
<!-- ============================================================ -->

<div style="border-bottom: 3px solid #4285F4; padding-bottom: 12px; margin-bottom: 20px;">

<div style="display: flex; align-items: center; justify-content: space-between;">
  <div>
    <img src="https://www.isae-supaero.fr/wp-content/uploads/2025/03/logo.svg" width="180">
  </div>
  <div style="text-align: right;">
    <img src="https://user-images.githubusercontent.com/63151412/167391313-4683cc69-2bf6-4597-b767-5c18e2bbbfa0.png" width="180">
  </div>
</div>

# Lab 03b — Online Endpoints & Load Testing — ✅ Solution

**Course:** MLOps Introductory Course on GCP · M2 Data Science · ISAE-SUPAERO  
**Lab created by:** Headmind Partners AI & Blockchain  
**Estimated duration:** ~1h15

</div>

## 📋 Lab Overview

In Lab 03a you trained a model and ran batch predictions — a pattern suited for offline workloads. But many applications require **real-time predictions** with low latency: a mobile app classifying a photo, a fraud detection system scoring transactions, or a recommendation engine serving results on page load. This lab focuses on the **online serving** side of MLOps.

Beyond basic deployment (which you practiced in Lab 02), production endpoints require **traffic management** (A/B testing, canary releases), **autoscaling**, and **load testing** to ensure they meet SLAs under heavy traffic.

### Learning Objectives

1. Deploy a model to a Vertex AI endpoint with **traffic splitting** and **autoscaling** configuration.
2. Send **online predictions** via the SDK and evaluate results.
3. Understand how to wrap a Vertex AI endpoint with **FastAPI** to build a custom API layer.
4. Run **load tests** with Locust and interpret throughput / latency metrics.
5. Reason about deployment strategies: canary, blue-green, and A/B testing.

### Notebook Structure

| # | Section | Focus |
|---|---------|-------|
| 0 | Setup | Install dependencies, imports, retrieve model from Lab 05 |
| 1 | Deploy to Endpoint | Create endpoint, deploy with traffic split & autoscaling |
| 2 | Online Predictions | Send requests via SDK, evaluate results |
| 3 | FastAPI Wrapper | Analyze the API layer, run locally, send requests |
| 4 | Load Testing with Locust | Run Locust, interpret results, discuss scaling |
| 5 | Cleanup | Undeploy model, delete endpoint |

### How to Read This Notebook

- **`# TODO`** — Code you need to write. Look for the `######` delimiters.
- **`✏️ Question`** — A conceptual question. Write your answer in the markdown cell below it.
- Cells **without** a TODO are provided — read them, run them, and make sure you understand them.
- Documentation links are provided in 📖 callouts whenever a new API is introduced.

---
## 0 · Setup

### 0.1 Install dependencies

In [ ]:
%pip install --upgrade --quiet google-cloud-aiplatform google-cloud-storage pillow numpy fastapi uvicorn locust

### 0.2 Imports

In [ ]:
import os
import json
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
from PIL import Image

from google.cloud import aiplatform

CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

print(f"Vertex AI SDK version: {aiplatform.__version__}")

### 0.3 Configuration & model retrieval

Use the same GCP project settings as Lab 05. Then retrieve the model you trained there.

In [ ]:
# ── Constants ──
PROJECT_ID = "your-project-id"          # @param {type:"string"}
LOCATION = "europe-west3"                # @param {type:"string"}
BUCKET_URI = "gs://your-bucket-name"    # @param {type:"string"}

##############################  TODO  ##############################
# Set your_name to a unique lowercase identifier (e.g. first letter of first name + last name).
your_name = "dupont"  # TODO
####################################################################

aiplatform.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=BUCKET_URI,
)
print(f"✅ Vertex AI initialized — project={PROJECT_ID}, location={LOCATION}")

In [ ]:
# ✅ SOLUTION
# Option B: List models and filter by display name
models = aiplatform.Model.list(filter='display_name="cifar10-model"')
model = models[0]

print(f"✅ Model loaded: {model.display_name} ({model.resource_name})")

---
## 1 · Deploy to an Online Endpoint

In Lab 02b, you created an endpoint and deployed a model with default settings. Here we go further: you will configure **traffic splitting** and **autoscaling** — two critical features for production deployments.

### 1.1 Deployment strategies

Before writing code, let's understand three common deployment strategies:

| Strategy | How it works | Use case |
|----------|-------------|----------|
| **Canary** | Route a small % of traffic to the new model, then gradually increase | Safe rollout — catch issues before they affect all users |
| **Blue-Green** | Two identical environments; switch traffic all at once | Zero-downtime releases with instant rollback |
| **A/B Testing** | Split traffic between models to compare performance on real data | Measure business impact of a new model |

Vertex AI supports these patterns through the `traffic_split` parameter when deploying models to an endpoint.

### 1.2 Create an endpoint

> 📖 **Docs:** [`Endpoint.create()`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.Endpoint#google_cloud_aiplatform_Endpoint_create)

In [ ]:
# ✅ SOLUTION
endpoint = aiplatform.Endpoint.create(
    display_name="cifar10-endpoint"
)

print(f"✅ Endpoint created: {endpoint.resource_name}")

### 1.3 Deploy with traffic split and autoscaling

When deploying, the `traffic_split` dictionary controls how incoming requests are distributed. The key `"0"` refers to the model being deployed *in this call*; existing deployed model IDs can be used as other keys.

**Autoscaling** is controlled by `min_replica_count` and `max_replica_count`. Vertex AI automatically adjusts the number of serving replicas between these bounds based on traffic load.

> 📖 **Docs:** [`Endpoint.deploy()`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.Endpoint#google_cloud_aiplatform_Endpoint_deploy)

In [ ]:
# ✅ SOLUTION
DEPLOYED_NAME = "cifar10-deployed"

response = endpoint.deploy(
    model=model,
    deployed_model_display_name=DEPLOYED_NAME,
    traffic_split={"0": 100},
    machine_type="n1-standard-4",
    min_replica_count=1,
    max_replica_count=3,
)

print(f"✅ Model deployed to endpoint.")

> ⏳ **Deployment takes 5–10 minutes.** While you wait, answer the question below.

**✏️ Question 1 — Traffic splitting & autoscaling**

a) You deployed with `traffic_split={"0": 100}`. Suppose you train an improved model and want to do a **canary release**, sending only 10% of traffic to the new model. How would you configure `traffic_split`? (Hint: you need the deployed model ID of the existing model.)

b) You set `max_replica_count=3`. What signals does Vertex AI use to decide when to scale up? What happens when traffic drops?

---
*✅ Solution:*

a) First, get the deployed model ID of the current model (e.g., `"1234567890"`). Then, when deploying the new model, set `traffic_split={"0": 10, "1234567890": 90}`. This sends 10% of traffic to the new model and 90% to the existing one. You can gradually increase the new model's share (e.g., 25%, 50%, 100%) as you gain confidence. If issues arise, you can route 100% back to the original model instantly.

b) Vertex AI monitors CPU utilization and request queue depth. When these metrics exceed internal thresholds, it provisions additional replicas (up to `max_replica_count`). When traffic drops, it scales back down to `min_replica_count`. The scaling is not instantaneous — there is a warm-up period as new replicas start and load the model. This is why `min_replica_count` should be at least 1 for always-on endpoints. Setting it to 0 enables "scale to zero" but introduces cold-start latency.

---

### 1.4 Inspect the deployment

In [ ]:
# Retrieve deployment details
deployed_models = endpoint.gca_resource.deployed_models
for dm in deployed_models:
    print(f"Deployed model ID:    {dm.id}")
    print(f"Display name:         {dm.display_name}")
    print(f"Model resource:       {dm.model}")
    print(f"Machine type:         {dm.dedicated_resources.machine_spec.machine_type}")
    print(f"Min replicas:         {dm.dedicated_resources.min_replica_count}")
    print(f"Max replicas:         {dm.dedicated_resources.max_replica_count}")
    print()

---
## 2 · Online Predictions

With the model deployed, we can send prediction requests via the SDK. Unlike batch prediction, online prediction returns results **synchronously** — the response comes back immediately.

### 2.1 Prepare test data

In [ ]:
# Download test images (if not already present from Lab 03a)
if not os.path.exists("cifar_test_images"):
    ! gsutil -m cp -r gs://cloud-samples-data/ai-platform-unified/cifar_test_images .

# Load and preprocess
IMAGE_DIRECTORY = "cifar_test_images"
image_files = [f for f in os.listdir(IMAGE_DIRECTORY) if f.endswith(".jpg")]
image_data = [np.asarray(Image.open(os.path.join(IMAGE_DIRECTORY, f))) for f in image_files]
x_test = [(img / 255.0).astype(np.float32).tolist() for img in image_data]
y_test = [int(f.split("_")[1]) for f in image_files]

print(f"✅ Loaded {len(x_test)} test images")

### 2.2 Send predictions

> 📖 **Docs:** [`Endpoint.predict()`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.Endpoint#google_cloud_aiplatform_Endpoint_predict)

In [ ]:
# ✅ SOLUTION
predictions = endpoint.predict(instances=x_test)

print(f"✅ Received {len(predictions.predictions)} predictions")

### 2.3 Evaluate results

In [ ]:
# ✅ SOLUTION
y_predicted = [np.argmax(pred) for pred in predictions.predictions]
correct = sum(np.array(y_predicted) == np.array(y_test))
accuracy = correct / len(y_test)

print(f"Accuracy: {accuracy:.0%} ({correct}/{len(y_test)})")
for i, (true, pred) in enumerate(zip(y_test, y_predicted)):
    status = "✅" if true == pred else "❌"
    print(f"  {status} Image {i}: true={CLASS_NAMES[true]}, predicted={CLASS_NAMES[pred]}")

**✏️ Question 2 — Batch vs. Online inference**

a) You just ran the same model with both batch and online inference. Compare the two approaches in terms of latency, throughput, cost, and appropriate use cases.

b) For the e-commerce scenario described in Lab 03a (classifying product images overnight), which approach is more appropriate and why?

---
*✅ Solution:*

a) Comparison:

| Dimension | Batch Prediction | Online Prediction |
|-----------|-----------------|-------------------|
| **Latency** | High (minutes to hours for the whole job) | Low (milliseconds per request) |
| **Throughput** | Very high — processes millions of records | Lower — designed for individual requests |
| **Cost** | Lower — resources provisioned only during the job | Higher — endpoint runs continuously |
| **Use case** | Periodic scoring, ETL pipelines, data enrichment | Real-time apps, user-facing features, fraud detection |

b) Batch prediction is clearly more appropriate: the workload runs once per night (no latency requirement), processes thousands of images in bulk (high throughput needed), and should be cost-efficient (no point paying for a 24/7 endpoint). You only pay for the compute time during the batch job, whereas an online endpoint would cost money even when idle overnight.

---

---
## 3 · FastAPI Wrapper

In production, you rarely expose a Vertex AI endpoint directly to end users. Instead, you build an **API layer** that handles authentication, input validation, response formatting, and business logic. FastAPI is a popular Python framework for building such APIs.

### 3.1 Why wrap a Vertex AI endpoint?

There are several reasons to add an API layer in front of your ML endpoint:
- **Authentication & authorization**: control who can access predictions
- **Input validation**: reject malformed requests before they reach the model
- **Response transformation**: convert raw model output into business-friendly responses (e.g., label names instead of class indices)
- **Rate limiting & caching**: protect the endpoint from abuse
- **Decoupling**: your application code depends on *your* API contract, not Vertex AI's

### 3.2 Analyze the API

Below is the FastAPI application that wraps the Vertex AI endpoint. Read through it and answer the question that follows.

> 📖 **Docs:** [FastAPI documentation](https://fastapi.tiangolo.com/)

In [ ]:
# Display the api.py file
print(open("api.py").read() if os.path.exists("api.py") else "api.py not found — please copy it to your working directory.")

If `api.py` is not present, create it with the cell below:

In [ ]:
%%writefile api.py
"""
FastAPI wrapper for the Vertex AI CIFAR-10 endpoint.

Usage:
  On Vertex AI Workbench:  Started programmatically from the notebook.
  On local machine:        uvicorn api:app --reload --port 8080
"""

from fastapi import FastAPI, HTTPException
import numpy as np
from pydantic import BaseModel
from google.cloud import aiplatform
from typing import List

app = FastAPI()

# ── Configuration ──
PROJECT_ID = "your-project-id"
LOCATION = "us-central1"
BUCKET_URI = "gs://your-bucket"
ENDPOINT_ID = "your-endpoint-id"

# ── Initialize Vertex AI ──
aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET_URI)
endpoint = aiplatform.Endpoint(ENDPOINT_ID)

# ── CIFAR-10 class names ──
CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

# ── Schemas ──
class PredictionRequest(BaseModel):
    """A single 32x32x3 image as a nested list of floats."""
    image: List[List[List[float]]]


class PredictionResponse(BaseModel):
    """The predicted class index, name, and full probability vector."""
    predicted_class: int
    class_name: str
    probabilities: List[float]


@app.get("/health")
def health_check():
    return {"status": "healthy"}


@app.post("/predict", response_model=PredictionResponse)
async def predict(request: PredictionRequest):
    try:
        result = endpoint.predict(instances=[request.image])
        probs = result.predictions[0]
        idx = int(np.argmax(probs))
        return PredictionResponse(
            predicted_class=idx,
            class_name=CLASS_NAMES[idx],
            probabilities=probs,
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Prediction error: {e}")

In [ ]:
# Patch api.py with your actual configuration
ENDPOINT_ID = endpoint.name  # The numeric endpoint ID

with open("api.py", "r") as f:
    content = f.read()

content = content.replace('PROJECT_ID = "your-project-id"', f'PROJECT_ID = "{PROJECT_ID}"')
content = content.replace('LOCATION = "us-central1"', f'LOCATION = "{LOCATION}"')
content = content.replace('BUCKET_URI = "gs://your-bucket"', f'BUCKET_URI = "{BUCKET_URI}"')
content = content.replace('ENDPOINT_ID = "your-endpoint-id"', f'ENDPOINT_ID = "{ENDPOINT_ID}"')

with open("api.py", "w") as f:
    f.write(content)

print(f"✅ api.py patched with ENDPOINT_ID={ENDPOINT_ID}")

### 3.3 Start the FastAPI server from the notebook

Since we are running on Vertex AI Workbench, we cannot easily open `localhost` in a browser. Instead, we start the FastAPI server **as a background process** and interact with it programmatically via `requests`.

> 💡 **Tip:** The server runs in a background thread so you can continue executing notebook cells.

In [ ]:
import subprocess, time, requests

# Start uvicorn as a background process
server_process = subprocess.Popen(
    ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8080"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Wait for the server to start
time.sleep(5)

# Health check
try:
    resp = requests.get("http://localhost:8080/health", timeout=10)
    print(f"✅ FastAPI server is running — health check: {resp.json()}")
except Exception as e:
    print(f"❌ Server not ready: {e}")
    print("   Check server logs with: server_process.stderr.read().decode()")

In [ ]:
# Test with a single image
single_image = x_test[0]
response = requests.post(
    "http://localhost:8080/predict",
    json={"image": single_image},
    timeout=30,
)

if response.status_code == 200:
    result = response.json()
    print(f"✅ Prediction: {result['class_name']} (class {result['predicted_class']})")
    print(f"   True label: {CLASS_NAMES[y_test[0]]}")
    print(f"   Top-3 probabilities: {sorted(enumerate(result['probabilities']), key=lambda x: -x[1])[:3]}")
else:
    print(f"❌ Error {response.status_code}: {response.text}")

**✏️ Question 3 — API design**

a) The original `api.py` from the old lab returned only the raw class index (an integer). The version above returns the class name and full probability vector. Why is returning richer responses a better practice for production APIs?

b) What is the purpose of the `/health` endpoint? In which deployment scenarios is it used?

c) Name two additional features you would add to this API before deploying it to production.

---
*✅ Solution:*

a) Returning richer responses is better because: clients don't need to maintain a separate mapping from class indices to names (reducing coupling); probabilities enable downstream logic like confidence thresholds (reject predictions below 70% confidence); and it makes debugging easier since you can see the full distribution.

b) The `/health` endpoint is used by load balancers, Kubernetes liveness/readiness probes, and monitoring systems to check if the service is running and ready to accept traffic. If the health check fails, the infrastructure can restart the container or stop routing traffic to it.

c) Two additions: (1) **Authentication** — API key or OAuth to control who can access the endpoint. (2) **Request logging** — log every prediction request and response for auditing, debugging, and monitoring model performance over time.

---

---
## 4 · Load Testing with Locust

Before going to production, you need to verify that your endpoint can handle the expected traffic. **Load testing** simulates concurrent users to measure throughput, latency, and failure rates under increasing load.

> 📖 **Docs:** [Locust](https://docs.locust.io/en/stable/)

### 4.1 How Locust works

Locust spawns virtual **users** that repeatedly execute tasks (in our case, sending prediction requests). Each user waits a random time between requests (the `wait_time`). Key metrics Locust reports:

- **RPS** (requests per second) — throughput.
- **Median latency** — the typical response time.
- **P95 latency** — 95% of requests are faster than this. Captures tail latency.
- **Failure rate** — percentage of requests that failed.

### 4.2 Headless mode

Locust normally provides a web UI at `localhost:8089`, but on Vertex AI Workbench you cannot access local ports in a browser. Instead, we use **headless mode** (`--headless`), which runs the load test from the command line and prints results to stdout. This works perfectly from a notebook cell.

> 💡 **Alternative — local VS Code setup:** If you want the full Locust web UI with interactive charts, see the appendix at the end of this section for instructions on running the notebook locally with VS Code.

In [ ]:
%%writefile locustfile.py
"""
Locust load test for the CIFAR-10 FastAPI wrapper.

Headless usage (from notebook):
    locust -f locustfile.py --host http://localhost:8080 --headless -u 5 -r 1 --run-time 60s --csv results

Web UI usage (from local machine):
    locust -f locustfile.py --host http://localhost:8080
"""

import os
import json
import numpy as np
from PIL import Image
from locust import HttpUser, task, between


def load_test_images():
    """Load and preprocess CIFAR-10 test images."""
    IMAGE_DIRECTORY = "cifar_test_images"
    image_files = sorted([f for f in os.listdir(IMAGE_DIRECTORY) if f.endswith(".jpg")])
    images = [np.asarray(Image.open(os.path.join(IMAGE_DIRECTORY, f))) for f in image_files]
    return [(img / 255.0).astype(np.float32).tolist() for img in images]


TEST_IMAGES = load_test_images()


class CifarUser(HttpUser):
    """Simulated user that sends prediction requests."""

    wait_time = between(1, 3)

    @task
    def predict_single_image(self):
        """Send a single random image for prediction."""
        idx = np.random.randint(len(TEST_IMAGES))
        payload = {"image": TEST_IMAGES[idx]}
        self.client.post("/predict", json=payload)

### 4.4 Run load tests

We will run three scenarios with increasing numbers of concurrent users. Each test runs for 60 seconds in headless mode. Locust writes CSV result files that we can analyze afterward.

The key Locust CLI flags:
- `--headless` — no web UI, run from the command line.
- `-u N` — number of concurrent users to simulate.
- `-r N` — spawn rate (users added per second).
- `--run-time 60s` — stop after 60 seconds.
- `--csv prefix` — write results to CSV files for analysis.

**Scenario 1 — Baseline (5 users)**

In [ ]:
# Run load test: 5 concurrent users for 60 seconds
! locust -f locustfile.py \
    --host http://localhost:8080 \
    --headless \
    -u 5 -r 1 \
    --run-time 60s \
    --csv results_5users \
    --only-summary

**Scenario 2 — Moderate load (15 users)**

In [ ]:
# Run load test: 15 concurrent users for 60 seconds
! locust -f locustfile.py \
    --host http://localhost:8080 \
    --headless \
    -u 15 -r 3 \
    --run-time 60s \
    --csv results_15users \
    --only-summary

**Scenario 3 — Heavy load (30 users)**

In [ ]:
# Run load test: 30 concurrent users for 60 seconds
! locust -f locustfile.py \
    --host http://localhost:8080 \
    --headless \
    -u 30 -r 5 \
    --run-time 60s \
    --csv results_30users \
    --only-summary

### 4.5 Analyze results

Locust writes CSV files with detailed statistics. Let's parse them and compare the three scenarios.

In [ ]:
import pandas as pd

scenarios = ["results_5users", "results_15users", "results_30users"]
summary_rows = []

for scenario in scenarios:
    stats_file = f"{scenario}_stats.csv"
    if os.path.exists(stats_file):
        df = pd.read_csv(stats_file)
        # Get the "Aggregated" row (summary of all endpoints)
        agg = df[df["Name"] == "Aggregated"]
        if not agg.empty:
            row = agg.iloc[0]
            summary_rows.append({
                "Scenario": scenario.replace("results_", ""),
                "Total Requests": int(row.get("Request Count", 0)),
                "Failures": int(row.get("Failure Count", 0)),
                "Median (ms)": int(row.get("Median Response Time", 0)),
                "P95 (ms)": int(row.get("95%", 0)),
                "Avg RPS": round(row.get("Requests/s", 0), 1),
            })
    else:
        print(f"⚠️ File not found: {stats_file}")

if summary_rows:
    results_df = pd.DataFrame(summary_rows)
    print(results_df.to_string(index=False))
else:
    print("No results found. Make sure the load tests ran successfully.")

**✏️ Question 4 — Load testing analysis**

a) How did the median latency and P95 latency change across the three scenarios? At what point (if any) did you observe failures or significant degradation?

b) Why is P95 latency more important than median latency for production SLAs?

c) Our endpoint has `max_replica_count=3`. How would the results change if we ran these same load tests after Vertex AI has had time to scale up? What if we set `max_replica_count=1`?

d) The load test goes through two layers: FastAPI → Vertex AI endpoint. Which layer do you think contributes more to the total latency? How would you isolate each layer's contribution?

---
*✅ Solution:*

a) Median latency stayed relatively stable for 5 and 15 users (the endpoint can handle the load), but increased significantly at 30 users. P95 latency increased more sharply, showing that tail latency is more sensitive to load. Failures may appear at 30 users if the endpoint is saturated.

b) P95 captures the experience of the worst-off users. A service with 200ms median but 5s P95 means 1 in 20 users waits 5 seconds — unacceptable for most applications. SLAs are typically defined on P95 or P99, not median.

c) With `max_replica_count=3` and time to scale, Vertex AI would add replicas under load, reducing latency at 30 users. With `max_replica_count=1`, there's no scaling possible — latency would degrade and failures would appear sooner.

d) The Vertex AI endpoint contributes most of the latency (model inference + network round-trip to GCP). FastAPI adds a thin layer of serialization/validation overhead (typically <10ms). To isolate: (1) test the Vertex AI endpoint directly using `endpoint.predict()` from the notebook and measure latency; (2) compare with the end-to-end latency through FastAPI.

---

**✏️ Question 5 — Direct endpoint vs. FastAPI wrapper**

In Section 4, we load-tested the FastAPI server, which in turn calls the Vertex AI endpoint. Alternatively, you could load-test the Vertex AI endpoint directly (bypassing FastAPI).

a) What are the pros and cons of testing through the FastAPI layer vs. testing the Vertex AI endpoint directly?

b) In a real production system, which approach gives you more useful data? Why?

---
*✅ Solution:*

a) **Through FastAPI — Pros:** tests the full production stack as users will experience it, including serialization, validation, and any business logic. **Cons:** adds latency from the FastAPI layer, making it harder to isolate the endpoint's raw performance.

**Direct endpoint — Pros:** measures the pure model serving performance without any wrapper overhead. **Cons:** doesn't reflect the real production architecture; may miss bottlenecks in the API layer (serialization, validation).

b) Testing through the full stack (FastAPI) gives more useful data because it reflects the actual user experience. However, you should also benchmark the raw endpoint separately to understand where latency comes from. The combination of both gives you the complete picture.

---

---
## 5 · Cleanup

Endpoints with deployed models incur charges as long as they are running. Always undeploy and delete when you are done.

> ⚠️ **Important:** Undeploy the model **before** deleting the endpoint.

In [ ]:
# List deployed models on the endpoint
deployed_models = endpoint.gca_resource.deployed_models
for dm in deployed_models:
    print(f"Deployed model ID: {dm.id}, Name: {dm.display_name}")

In [ ]:
# ✅ SOLUTION
for dm in deployed_models:
    endpoint.undeploy(deployed_model_id=dm.id)
    print(f"✅ Model {dm.id} undeployed.")

endpoint.delete()
print("✅ Endpoint deleted.")

In [ ]:
# Optionally delete the model (only if you no longer need it)
model.delete()
print("✅ Model deleted.")

---
## Summary

In this lab, you learned to:

| Step | What you did | Tool / Feature used |
|------|-------------|---------------------|
| Deploy | Deployed a model with traffic split and autoscaling | `Endpoint.deploy()`, `traffic_split`, `min/max_replica_count` |
| Online Predictions | Sent real-time requests via SDK | `Endpoint.predict()` |
| FastAPI Wrapper | Built a custom API layer in front of Vertex AI | FastAPI, Pydantic, `uvicorn` |
| Load Testing | Simulated concurrent users and measured latency | Locust, `locustfile.py` |
| Cleanup | Undeployed model and deleted endpoint | `endpoint.undeploy()`, `endpoint.delete()` |

**Next lab:** In the next session, you will build an **ML pipeline** with KFP to automate the entire training → evaluation → deployment workflow.